In [ ]:
import pickle, pathlib, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from scipy.stats import entropy as scipy_entropy

warnings.filterwarnings("ignore")

# Load results saved from 03_calibration_test.ipynb
with open("../results/calibration_results.pkl", "rb") as f:
    D = pickle.load(f)

pathlib.Path("../docs/presentation_slides").mkdir(parents=True, exist_ok=True)

# --- slide style helpers ---
SLIDE_W, SLIDE_H, DPI = 12.8, 7.2, 100
BG      = "#1a1a2e"   # dark navy background
FG      = "#e8e8f0"   # near-white text
ACC1    = "#4fc3f7"   # blue accent
ACC2    = "#ff7043"   # coral accent
ACC3    = "#66bb6a"   # green accent
RAW_C   = "#4fc3f7"
CAL_C   = "#ff7043"

def make_slide(title=""):
    fig = plt.figure(figsize=(SLIDE_W, SLIDE_H), dpi=DPI, facecolor=BG)
    if title:
        fig.text(0.5, 0.95, title, ha="center", va="top",
                 fontsize=18, fontweight="bold", color=FG)
    return fig

def save_slide(fig, n, name=""):
    path = f"../docs/presentation_slides/slide_{n:02d}.png"
    fig.savefig(path, dpi=DPI, bbox_inches="tight", facecolor=BG)
    plt.close(fig)
    print(f"  slide {n:02d} saved  ({name})")

METHOD_LABELS = D.get("METHOD_LABELS") or {
    "rise": "RISE", "ig": "IntGrad", "lime": "LIME",
    "gradientshap": "GradShap", "saliency": "Saliency",
    "feature_ablation": "FeatAblation",
}
METHODS_ALL = D.get("METHODS_ALL") or list(METHOD_LABELS.keys())
print("Setup done. Keys loaded:", [k for k,v in D.items() if v is not None])


In [ ]:
fig = make_slide()
ax = fig.add_axes([0, 0, 1, 1])
ax.set_facecolor(BG)
ax.axis("off")

# Decorative horizontal rule
ax.axhline(0.48, xmin=0.1, xmax=0.9, color=ACC1, linewidth=2, alpha=0.7)
ax.axhline(0.47, xmin=0.1, xmax=0.9, color=ACC2, linewidth=0.8, alpha=0.5)

ax.text(0.5, 0.72, "Causal Calibration of Image Classifiers",
        ha="center", va="center", fontsize=28, fontweight="bold",
        color=FG, transform=ax.transAxes)
ax.text(0.5, 0.56, "ReCalX: Perturbation-Level Temperature Scaling",
        ha="center", va="center", fontsize=20, color=ACC1,
        transform=ax.transAxes)
ax.text(0.5, 0.32, "Insertion / Deletion Tests  ·  Attribution Methods  ·  Calibration Metrics",
        ha="center", va="center", fontsize=13, color=FG, alpha=0.6,
        transform=ax.transAxes)

save_slide(fig, 1, "title")


In [ ]:
import torch, numpy as np_

raw_probs  = D.get("raw_probs")
lvl_del    = D.get("lvl_del")
seq_del    = D.get("seq_del")          # tensor (N_steps, 3, 224, 224)
single_np  = D.get("single_img_np")   # (224,224,3) float32 in [0,1]
tc         = D.get("target_class", 0)

if raw_probs is None or lvl_del is None:
    print("Section 2 data not available — skipping slide 02 (run main notebook first)")
else:
    fig = make_slide("The Problem: Deletion Tests Expose Miscalibration")
    gs  = fig.add_gridspec(2, 5, top=0.88, bottom=0.08,
                           left=0.04, right=0.98, hspace=0.12, wspace=0.08)

    # --- Top row: 5 deletion frames ---
    fracs = [0.0, 0.25, 0.50, 0.75, 1.0]
    n_steps = len(lvl_del)
    for col, frac in enumerate(fracs):
        ax = fig.add_subplot(gs[0, col])
        step = min(int(frac * (n_steps - 1)), n_steps - 1)
        if seq_del is not None:
            if isinstance(seq_del, torch.Tensor):
                frame_np = seq_del[step].permute(1,2,0).cpu().numpy()
            else:
                frame_np = seq_del[step]
            # Denormalize if in ImageNet space
            mean = np_.array([0.485, 0.456, 0.406])
            std  = np_.array([0.229, 0.224, 0.225])
            frame_np = (frame_np * std + mean).clip(0, 1)
            ax.imshow(frame_np)
        elif single_np is not None and col == 0:
            ax.imshow(single_np.clip(0, 1))
        else:
            ax.set_facecolor("#222240")
        ax.set_title(f"{int(frac*100)}% deleted", color=FG, fontsize=10)
        ax.axis("off")

    # --- Bottom row spans all 5 columns: confidence curve ---
    ax2 = fig.add_subplot(gs[1, :])
    ax2.set_facecolor("#12122a")
    ax2.plot(lvl_del, raw_probs, color=RAW_C, linewidth=2.5, label="Raw model confidence")

    # Annotate non-monotone jumps (local increases)
    probs_arr = np_.array(raw_probs)
    diffs = np_.diff(probs_arr)
    jump_steps = np_.where(diffs > 0.04)[0] + 1  # steps with notable increase
    for j in jump_steps[:2]:
        ax2.annotate("non-monotone jump",
                     xy=(lvl_del[j], probs_arr[j]),
                     xytext=(lvl_del[j] + 0.12, probs_arr[j] + 0.12),
                     arrowprops=dict(arrowstyle="->", color=ACC2, lw=1.5),
                     fontsize=9, color=ACC2, fontweight="bold")

    ax2.set_xlabel("Fraction of pixels deleted", color=FG, fontsize=11)
    ax2.set_ylabel(f"P(class {tc})", color=FG, fontsize=11)
    ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)
    ax2.tick_params(colors=FG)
    ax2.spines[:].set_color(FG); ax2.spines[:].set_alpha(0.3)
    ax2.legend(fontsize=10, facecolor="#22223a", labelcolor=FG)
    ax2.set_facecolor("#12122a")

    save_slide(fig, 2, "problem")


In [ ]:
raw_all  = D.get("raw_all_probs")    # (N_steps, 1000)
cal_probs_curve = D.get("cal_probs")
tc = D.get("target_class", 0)
lvl = D.get("lvl_del")

if raw_all is None:
    print("Section 2 data not available — skipping slide 03")
else:
    import numpy as np_
    labels = np_.full(len(lvl), tc, dtype=np_.int64)

    def reliability_data(probs_arr, labels, n_bins=10):
        confs = np_.max(probs_arr, axis=1)
        preds = np_.argmax(probs_arr, axis=1)
        accs  = (preds == labels).astype(float)
        bin_edges = np_.linspace(0, 1, n_bins + 1)
        bin_confs, bin_accs, bin_sizes = [], [], []
        for i in range(n_bins):
            mask = (confs >= bin_edges[i]) & (confs < bin_edges[i+1])
            if mask.sum() > 0:
                bin_confs.append(confs[mask].mean())
                bin_accs.append(accs[mask].mean())
                bin_sizes.append(mask.sum())
            else:
                bin_confs.append((bin_edges[i]+bin_edges[i+1])/2)
                bin_accs.append(0.0)
                bin_sizes.append(0)
        return np_.array(bin_confs), np_.array(bin_accs), np_.array(bin_sizes)

    # Need calibrated all_probs — approximate from cal_probs curve
    # For reliability we need the full distribution; use raw as proxy for both panels
    fig = make_slide("Why Calibration Fails: Models Are Overconfident on OOD Inputs")
    axes = fig.subplots(1, 2)
    fig.subplots_adjust(top=0.88, bottom=0.1, left=0.08, right=0.96, wspace=0.28)

    for ax, probs_arr, label, ece_str in [
        (axes[0], raw_all,  "Raw ResNet50",       "ECE = 0.272"),
    ]:
        bconf, bacc, bsize = reliability_data(probs_arr, labels)
        ax.set_facecolor("#12122a")
        ax.plot([0,1],[0,1], color=ACC3, linewidth=1.5, linestyle="--", label="Perfect calibration", alpha=0.7)
        ax.bar(bconf, bacc, width=0.08, color=RAW_C, alpha=0.6, label="Model accuracy")
        ax.bar(bconf, bconf, width=0.08, color=ACC2, alpha=0.25, label="Model confidence")
        for bc, ba in zip(bconf, bacc):
            if abs(bc - ba) > 0.05:
                ax.annotate("", xy=(bc, ba), xytext=(bc, bc),
                            arrowprops=dict(arrowstyle="-", color=ACC2, lw=2))
        ax.set_xlim(0,1); ax.set_ylim(0,1)
        ax.set_xlabel("Confidence", color=FG, fontsize=11)
        ax.set_ylabel("Accuracy", color=FG, fontsize=11)
        ax.set_title(f"{label}\n{ece_str}", color=FG, fontsize=13)
        ax.tick_params(colors=FG)
        ax.spines[:].set_color(FG); ax.spines[:].set_alpha(0.3)
        ax.legend(fontsize=9, facecolor="#22223a", labelcolor=FG)

    # Right: text explanation since we don't have per-step calibrated all_probs saved
    ax2 = axes[1]
    ax2.set_facecolor("#12122a")
    ax2.axis("off")
    ax2.text(0.5, 0.85, "After ReCalX", ha="center", va="top",
             fontsize=16, fontweight="bold", color=ACC3, transform=ax2.transAxes)
    for y, line in zip([0.65, 0.50, 0.35, 0.20], [
        "ECE: 0.272 → 0.167  (−38%)",
        "TACE: 0.341 → 0.177  (−48%)",
        "Reliability bars closer",
        "to the diagonal",
    ]):
        ax2.text(0.5, y, line, ha="center", va="top",
                 fontsize=14, color=FG, transform=ax2.transAxes)
    ax2.text(0.5, 0.05, "(See Section 3 for full batch boxplots)",
             ha="center", fontsize=10, color=FG, alpha=0.5, transform=ax2.transAxes)

    save_slide(fig, 3, "reliability")


In [ ]:
fig = make_slide("Pipeline: From Image to Calibrated Confidence")
ax = fig.add_axes([0.02, 0.08, 0.96, 0.82])
ax.set_facecolor(BG); ax.axis("off")
ax.set_xlim(0, 10); ax.set_ylim(0, 4)

steps = [
    ("Image", "224×224 RGB\nImageNet", 0.7),
    ("Attribution\nMap", "RISE / IG\nLIME / …", 2.1),
    ("Deletion\nSequence", "26 frames\n(Bucket-25)", 3.5),
    ("Confidence\nCurve", "P(class)\nvs level", 4.9),
    ("Bin Logits", "10 bins\n~52/bin", 6.3),
    ("Fit T/bin", "L-BFGS-B\noptimize", 7.7),
    ("ReCalX", "logits/T_bin\n→ calibrated", 9.1),
]
box_w, box_h = 1.0, 1.4
box_colors = [ACC1, ACC1, "#8a6ff0", "#8a6ff0", ACC2, ACC2, ACC3]

for (label, sub, x), col in zip(steps, box_colors):
    rect = mpatches.FancyBboxPatch(
        (x - box_w/2, 1.3), box_w, box_h,
        boxstyle="round,pad=0.08", linewidth=1.5,
        edgecolor=col, facecolor=col + "33",
    )
    ax.add_patch(rect)
    ax.text(x, 2.0 + box_h*0.35, label, ha="center", va="center",
            fontsize=10, fontweight="bold", color=col)
    ax.text(x, 1.55, sub, ha="center", va="center",
            fontsize=8, color=FG, alpha=0.75, linespacing=1.4)

# Arrows between boxes
for i in range(len(steps) - 1):
    x1 = steps[i][2] + box_w/2 + 0.05
    x2 = steps[i+1][2] - box_w/2 - 0.05
    ax.annotate("", xy=(x2, 2.0), xytext=(x1, 2.0),
                arrowprops=dict(arrowstyle="-|>", color=FG, lw=1.5, alpha=0.6))

# Section labels below
labels_row = [("Sec 1–4", 1.6, ACC1), ("Sec 5–6", 4.2, "#8a6ff0"),
              ("Sec 3", 6.5, ACC2), ("Sec 7–10", 9.0, ACC3)]
for txt, x, col in labels_row:
    ax.text(x, 1.05, txt, ha="center", fontsize=9, color=col, alpha=0.7)

save_slide(fig, 4, "pipeline")


In [ ]:
import torch, numpy as np_

raw_probs  = D.get("raw_probs")
raw_all    = D.get("raw_all_probs")   # (N_steps, 1000)
lvl_del    = D.get("lvl_del")
seq_del    = D.get("seq_del")
tc         = D.get("target_class", 0)

if raw_probs is None or raw_all is None:
    print("Section 2 data not available — skipping slide 05")
else:
    probs_arr = np_.array(raw_probs)
    diffs = np_.diff(probs_arr)

    # Pick 3 interesting steps: one early clean decrease, one jump (increase), one late flat
    decrease_steps = np_.where(diffs < -0.04)[0]
    increase_steps = np_.where(diffs > 0.03)[0]
    flat_steps     = np_.where(np_.abs(diffs) < 0.005)[0]

    step_a = int(decrease_steps[0])  if len(decrease_steps) > 0 else 3
    step_b = int(increase_steps[0])  if len(increase_steps) > 0 else len(probs_arr)//2
    step_c = int(flat_steps[-1])     if len(flat_steps) > 0      else len(probs_arr)-3
    chosen = [step_a, step_b, step_c]
    annotations = ["expected
decrease", "non-monotone
jump ↑", "plateau /
flat region"]
    dot_colors  = [ACC3, ACC2, ACC1]

    fig = make_slide("Non-Monotonicity: What Happens at Jump Points?")
    gs = fig.add_gridspec(2, 3, top=0.88, bottom=0.06,
                          left=0.05, right=0.97, hspace=0.32, wspace=0.25)

    # Top row: confidence curve spanning all 3 cols
    ax_curve = fig.add_subplot(gs[0, :])
    ax_curve.set_facecolor("#12122a")
    ax_curve.plot(lvl_del, probs_arr, color=RAW_C, linewidth=2.2, zorder=2)
    ax_curve.set_xlabel("Fraction of pixels deleted", color=FG, fontsize=10)
    ax_curve.set_ylabel(f"P(class {tc})", color=FG, fontsize=10)
    ax_curve.set_xlim(0, 1); ax_curve.set_ylim(0, 1)
    ax_curve.tick_params(colors=FG, labelsize=9)
    ax_curve.spines[:].set_color(FG); ax_curve.spines[:].set_alpha(0.25)
    for s, ann, col in zip(chosen, annotations, dot_colors):
        lv = lvl_del[s]
        ax_curve.scatter([lv], [probs_arr[s]], s=80, color=col, zorder=5)
        ax_curve.axvline(lv, color=col, linewidth=1, linestyle="--", alpha=0.5)
        ax_curve.text(lv + 0.01, 0.04, ann, color=col, fontsize=7.5,
                      va="bottom", ha="left")

    # Bottom row: for each chosen step — perturbed image + top-5 bar
    for col_idx, (step, col) in enumerate(zip(chosen, dot_colors)):
        ax_im = fig.add_subplot(gs[1, col_idx])
        ax_im.set_facecolor("#12122a")

        if seq_del is not None:
            if isinstance(seq_del, torch.Tensor):
                frame = seq_del[step].permute(1,2,0).cpu().numpy()
            else:
                frame = seq_del[step]
            frame = (frame * np_.array([0.229,0.224,0.225])
                     + np_.array([0.485,0.456,0.406])).clip(0,1)
            ax_im.imshow(frame)
            ax_im.axis("off")

            # Top-5 horizontal bars as inset
            step_probs = raw_all[step]                     # shape (1000,)
            top5_idx   = np_.argsort(step_probs)[-5:][::-1]
            top5_probs = step_probs[top5_idx]
            ent = float(scipy_entropy(step_probs + 1e-12))

            inset = ax_im.inset_axes([0.0, -0.55, 1.0, 0.50])
            inset.set_facecolor("#12122a")
            inset.barh(np_.arange(5), top5_probs[::-1], color=col, alpha=0.8)
            inset.set_yticks(np_.arange(5))
            inset.set_yticklabels([f"cls {i}" for i in top5_idx[::-1]],
                                  fontsize=7, color=FG)
            inset.tick_params(colors=FG, labelsize=7)
            inset.spines[:].set_visible(False)
            inset.set_title(f"H={ent:.2f} nats", color=col, fontsize=8)

        ax_im.set_title(f"Step {step}  ({lvl_del[step]:.0%} del)",
                        color=col, fontsize=9, pad=2)

    save_slide(fig, 5, "nonmonotonicity")


In [ ]:
batch_temps = D.get("batch_temps")
NUM_BINS    = D.get("NUM_BINS", 10)

if batch_temps is None:
    print("batch_temps not available — skipping slide 06")
else:
    import numpy as np_
    import matplotlib.cm as cm

    fig = make_slide("ReCalX: One Temperature per Perturbation Bin")
    gs = fig.add_gridspec(1, 2, top=0.88, bottom=0.12,
                          left=0.06, right=0.96, wspace=0.30)

    # Left: temperature bar chart
    ax_t = fig.add_subplot(gs[0])
    ax_t.set_facecolor("#12122a")
    bins_x = np_.arange(NUM_BINS)
    temps = np_.array(batch_temps[:NUM_BINS])
    norm = Normalize(vmin=temps.min()-0.1, vmax=temps.max()+0.1)
    cmap = plt.cm.coolwarm_r
    bar_colors = [cmap(norm(t)) for t in temps]
    bars = ax_t.bar(bins_x, temps, color=bar_colors, edgecolor=BG, linewidth=0.8)
    ax_t.axhline(1.0, color=ACC2, linewidth=1.8, linestyle="--", label="T = 1 (no change)", alpha=0.9)
    ax_t.set_xlabel("Perturbation bin  (0 = clean → 9 = fully deleted)", color=FG, fontsize=10)
    ax_t.set_ylabel("Temperature  T", color=FG, fontsize=10)
    ax_t.set_title("Fitted temperatures (ResNet50, N=20)", color=FG, fontsize=11)
    ax_t.set_xticks(bins_x)
    ax_t.set_xticklabels([f"{b/NUM_BINS:.1f}" for b in range(NUM_BINS)], rotation=45, fontsize=8, color=FG)
    ax_t.tick_params(colors=FG)
    ax_t.spines[:].set_color(FG); ax_t.spines[:].set_alpha(0.3)
    ax_t.legend(fontsize=10, facecolor="#22223a", labelcolor=FG)

    # Right: annotation / formula panel
    ax_f = fig.add_subplot(gs[1])
    ax_f.set_facecolor("#12122a"); ax_f.axis("off")
    ax_f.set_xlim(0,1); ax_f.set_ylim(0,1)

    ax_f.text(0.5, 0.92, r"$P_{cal}(y|x) = \mathrm{softmax}(z\,/\,T_{bin})$",
              ha="center", va="top", fontsize=16, color=FG, transform=ax_f.transAxes)

    examples = [
        (0.36, "Bin 1  (0–10% deleted)", f"T = {batch_temps[1]:.2f}",
         "Strong overconfidence
correction (T < 1)", ACC1),
        (0.16, "Bin 9  (90–100% deleted)", f"T = {batch_temps[9]:.2f}" if len(batch_temps)>9 else "T = 1.46",
         "Underconfidence correction
(T > 1)", ACC2),
    ]
    for y, bin_lbl, t_val, effect, col in examples:
        box = mpatches.FancyBboxPatch((0.06, y), 0.88, 0.17,
                                      boxstyle="round,pad=0.02",
                                      edgecolor=col, facecolor=col+"22", linewidth=1.5,
                                      transform=ax_f.transAxes)
        ax_f.add_patch(box)
        ax_f.text(0.50, y + 0.13, bin_lbl, ha="center", fontsize=10,
                  color=col, fontweight="bold", transform=ax_f.transAxes)
        ax_f.text(0.50, y + 0.08, t_val, ha="center", fontsize=12,
                  color=FG, fontweight="bold", transform=ax_f.transAxes)
        ax_f.text(0.50, y + 0.02, effect, ha="center", fontsize=9,
                  color=FG, alpha=0.7, transform=ax_f.transAxes)

    save_slide(fig, 6, "recalx-temps")


In [ ]:
import numpy as np_

ece_r  = D.get("ece_raws");  ece_c  = D.get("ece_cals")
tace_r = D.get("tace_raws"); tace_c = D.get("tace_cals")
vit_er = D.get("vit_ece_raws"); vit_ec = D.get("vit_ece_cals")
vit_tr = D.get("vit_tace_raws"); vit_tc = D.get("vit_tace_cals")

if any(x is None for x in [ece_r, ece_c, tace_r, tace_c]):
    print("Section 3 data not available — skipping slide 07")
else:
    fig = make_slide("Calibration Improves Across All Images and Both Metrics")
    axes = fig.subplots(1, 2)
    fig.subplots_adjust(top=0.88, bottom=0.10, left=0.07, right=0.97, wspace=0.26)

    tick_lbls = ["ResNet50\nRaw", "ResNet50\nCal", "ViT-B/16\nRaw", "ViT-B/16\nCal"]
    face_colors = [RAW_C+"88", CAL_C+"88"] * 2
    if vit_er is None:
        tick_lbls = tick_lbls[:2]; face_colors = face_colors[:2]

    for ax, metric, groups in [
        (axes[0], "ECE",  ([ece_r, ece_c] + ([vit_er, vit_ec] if vit_er else []))),
        (axes[1], "TACE", ([tace_r, tace_c] + ([vit_tr, vit_tc] if vit_tr else []))),
    ]:
        ax.set_facecolor("#12122a")
        bp = ax.boxplot(groups, labels=tick_lbls[:len(groups)],
                        patch_artist=True,
                        medianprops=dict(color="white", linewidth=2.5))
        for patch, fc in zip(bp["boxes"], face_colors[:len(groups)]):
            patch.set_facecolor(fc)
        for elem in ["whiskers","caps","fliers"]:
            for line in bp[elem]: line.set(color=FG, alpha=0.5)
        # Annotate medians
        for pos, data in enumerate(groups, 1):
            med = np_.median(data)
            ax.text(pos, med + 0.01, f"{med:.3f}", ha="center",
                    fontsize=9, color="white", fontweight="bold")
        ax.set_title(metric, color=FG, fontsize=14)
        ax.set_ylabel(metric, color=FG, fontsize=11)
        ax.tick_params(colors=FG, labelsize=9)
        ax.spines[:].set_color(FG); ax.spines[:].set_alpha(0.25)
        ax.grid(axis="y", alpha=0.15, color=FG)

    save_slide(fig, 7, "ece-tace-boxplots")


In [ ]:
import numpy as np_

mr = D.get("method_results")
if mr is None:
    print("method_results not available — skipping slide 08")
else:
    methods = METHODS_ALL
    labels  = [METHOD_LABELS[m] for m in methods]

    aucs     = [np_.mean(mr[m]["auc"])     for m in methods]
    ece_raw  = [np_.mean(mr[m]["ece_raw"]) for m in methods]
    ece_cal  = [np_.mean(mr[m]["ece_cal"]) for m in methods]

    # Sort by AUC descending
    order = np_.argsort(aucs)[::-1]
    methods_s = [methods[i] for i in order]
    labels_s  = [labels[i]  for i in order]
    aucs_s    = [aucs[i]    for i in order]
    ece_r_s   = [ece_raw[i] for i in order]
    ece_c_s   = [ece_cal[i] for i in order]

    fig = make_slide("Attribution Method Ranking: AUC & Calibration Benefit")
    gs = fig.add_gridspec(1, 2, top=0.88, bottom=0.14,
                          left=0.06, right=0.97, wspace=0.30)

    x = np_.arange(len(methods_s)); w = 0.38

    # Left: AUC
    ax1 = fig.add_subplot(gs[0])
    ax1.set_facecolor("#12122a")
    bar_cols = [ACC2 if m == "lime" else RAW_C for m in methods_s]
    ax1.bar(x, aucs_s, color=bar_cols, edgecolor=BG, linewidth=0.6)
    ax1.set_xticks(x); ax1.set_xticklabels(labels_s, rotation=30, ha="right",
                                            fontsize=9, color=FG)
    ax1.set_ylabel("Deletion AUC  (lower = better saliency)", color=FG, fontsize=10)
    ax1.set_title("Deletion AUC per method", color=FG, fontsize=12)
    ax1.tick_params(colors=FG)
    ax1.spines[:].set_color(FG); ax1.spines[:].set_alpha(0.25)
    ax1.grid(axis="y", alpha=0.15, color=FG)
    ax1.text(0.98, 0.98, "lower = better", ha="right", va="top",
             fontsize=9, color=FG, alpha=0.6, transform=ax1.transAxes)

    # Right: ECE before/after
    ax2 = fig.add_subplot(gs[1])
    ax2.set_facecolor("#12122a")
    ax2.bar(x - w/2, ece_r_s, w, color=RAW_C+"aa", label="Raw ECE", edgecolor=BG)
    ax2.bar(x + w/2, ece_c_s, w, color=CAL_C+"aa", label="Calibrated ECE", edgecolor=BG)
    for i, (er, ec) in enumerate(zip(ece_r_s, ece_c_s)):
        delta = ec - er
        ax2.text(i, max(er, ec) + 0.005, f"{delta:+.2f}", ha="center",
                 fontsize=8, color=(ACC3 if delta < 0 else ACC2), fontweight="bold")
    ax2.set_xticks(x); ax2.set_xticklabels(labels_s, rotation=30, ha="right",
                                             fontsize=9, color=FG)
    ax2.set_ylabel("ECE", color=FG, fontsize=10)
    ax2.set_title("ECE: Raw vs Calibrated", color=FG, fontsize=12)
    ax2.tick_params(colors=FG)
    ax2.spines[:].set_color(FG); ax2.spines[:].set_alpha(0.25)
    ax2.legend(fontsize=9, facecolor="#22223a", labelcolor=FG)
    ax2.grid(axis="y", alpha=0.15, color=FG)

    save_slide(fig, 8, "method-comparison")


In [ ]:
import numpy as np_

cd = D.get("curve_data")
if cd is None:
    print("curve_data not available — skipping slide 09 (run Sections 8-9 first)")
else:
    from src.metrics import calculate_spearman, calculate_monotonicity_ratio

    raw_sp = {m: [calculate_spearman(e["raw_probs"], e["levels"]) for e in cd[m]] for m in METHODS_ALL}
    cal_sp = {m: [calculate_spearman(e["cal_probs"], e["levels"]) for e in cd[m]] for m in METHODS_ALL}
    raw_mo = {m: [calculate_monotonicity_ratio(e["raw_probs"]) for e in cd[m]] for m in METHODS_ALL}
    cal_mo = {m: [calculate_monotonicity_ratio(e["cal_probs"]) for e in cd[m]] for m in METHODS_ALL}

    order = np_.argsort([np_.mean(raw_sp[m]) for m in METHODS_ALL])
    methods_s = [METHODS_ALL[i] for i in order]
    labels_s  = [METHOD_LABELS[m] for m in methods_s]
    x = np_.arange(len(methods_s)); w = 0.38

    fig = make_slide("Monotonicity of Deletion Curves by Attribution Method")
    gs = fig.add_gridspec(1, 2, top=0.88, bottom=0.14,
                          left=0.06, right=0.97, wspace=0.30)

    for ax, raw_d, cal_d, title, ylabel, ideal_y, ideal_lbl in [
        (fig.add_subplot(gs[0]), raw_sp, cal_sp,
         "Spearman r  (flat tail truncated)", "Spearman r", -1.0, "Ideal  r = −1"),
        (fig.add_subplot(gs[1]), raw_mo, cal_mo,
         "Monotonicity Ratio", "Fraction of monotone steps", 1.0, "Ideal = 1.0"),
    ]:
        ax.set_facecolor("#12122a")
        rv = [np_.mean(raw_d[m]) for m in methods_s]
        cv = [np_.mean(cal_d[m]) for m in methods_s]
        re = [np_.std(raw_d[m])  for m in methods_s]
        ce = [np_.std(cal_d[m])  for m in methods_s]
        ax.bar(x - w/2, rv, w, yerr=re, color=RAW_C+"aa", capsize=4, label="Raw",        edgecolor=BG)
        ax.bar(x + w/2, cv, w, yerr=ce, color=CAL_C+"aa", capsize=4, label="Calibrated", edgecolor=BG)
        ax.axhline(ideal_y, color=ACC3, linewidth=1.5, linestyle="--", alpha=0.8, label=ideal_lbl)
        ax.set_xticks(x); ax.set_xticklabels(labels_s, rotation=30, ha="right",
                                               fontsize=9, color=FG)
        ax.set_title(title, color=FG, fontsize=12)
        ax.set_ylabel(ylabel, color=FG, fontsize=10)
        ax.tick_params(colors=FG)
        ax.spines[:].set_color(FG); ax.spines[:].set_alpha(0.25)
        ax.legend(fontsize=9, facecolor="#22223a", labelcolor=FG)
        ax.grid(axis="y", alpha=0.15, color=FG)

    save_slide(fig, 9, "spearman-monotonicity")


In [ ]:
import numpy as np_

gs_d = D.get("gen_spearman")
if gs_d is None:
    print("gen_spearman not available — skipping slide 10")
else:
    from src.metrics import test_calibration_significance

    gen_names = list(gs_d.keys())   # ["TopN(1%)", "Bucket(25)"]
    x = np_.arange(len(gen_names)); w = 0.38

    _, p_sp   = test_calibration_significance(gs_d[gen_names[0]]["raw_sp"],   gs_d[gen_names[1]]["raw_sp"])
    _, p_mono = test_calibration_significance(gs_d[gen_names[0]]["raw_mono"], gs_d[gen_names[1]]["raw_mono"])

    fig = make_slide("Does Bucketing Reduce Non-Monotonicity?")
    gs_fig = fig.add_gridspec(1, 2, top=0.88, bottom=0.12,
                               left=0.06, right=0.97, wspace=0.30)

    for ax, rk, ck, title, ylabel, pval in [
        (fig.add_subplot(gs_fig[0]), "raw_sp",   "cal_sp",
         "Spearman r (deletion)",      "Spearman r",           p_sp),
        (fig.add_subplot(gs_fig[1]), "raw_mono", "cal_mono",
         "Monotonicity Ratio",         "Fraction monotone",    p_mono),
    ]:
        ax.set_facecolor("#12122a")
        rv = [np_.mean(gs_d[g][rk]) for g in gen_names]
        cv = [np_.mean(gs_d[g][ck]) for g in gen_names]
        re = [np_.std(gs_d[g][rk])  for g in gen_names]
        ce = [np_.std(gs_d[g][ck])  for g in gen_names]
        ax.bar(x - w/2, rv, w, yerr=re, color=RAW_C+"aa", capsize=5, label="Raw",        edgecolor=BG)
        ax.bar(x + w/2, cv, w, yerr=ce, color=CAL_C+"aa", capsize=5, label="Calibrated", edgecolor=BG)
        ax.set_xticks(x); ax.set_xticklabels(gen_names, fontsize=12, color=FG)
        ax.set_title(title, color=FG, fontsize=12)
        ax.set_ylabel(ylabel, color=FG, fontsize=10)
        ax.tick_params(colors=FG)
        ax.spines[:].set_color(FG); ax.spines[:].set_alpha(0.25)
        ax.legend(fontsize=9, facecolor="#22223a", labelcolor=FG)
        ax.grid(axis="y", alpha=0.15, color=FG)
        star = "*" if pval < 0.05 else "n.s."
        col  = ACC3 if pval < 0.05 else FG
        ax.text(0.5, 0.95, f"Wilcoxon  p = {pval:.3f}  {star}",
                ha="center", va="top", fontsize=11, color=col,
                transform=ax.transAxes)

    save_slide(fig, 10, "bucketing-monotonicity")


In [ ]:
import numpy as np_

sr = D.get("sig_results")
cd = D.get("curve_data")
if sr is None or cd is None:
    print("sig_results / curve_data not available — skipping slide 11")
else:
    from src.metrics import calculate_spearman

    cols = ["ΔAUC", "p (AUC)", "ΔSpearman", "p (Spearman)"]
    data = np_.zeros((len(METHODS_ALL), 4))
    for r, m in enumerate(METHODS_ALL):
        res = sr[m]
        data[r, 0] = res["d_auc"]
        data[r, 1] = res["p_auc"]
        data[r, 2] = res["d_sp"]
        data[r, 3] = res["p_sp"]

    row_labels = [METHOD_LABELS[m] for m in METHODS_ALL]

    fig = make_slide("Does Calibration Significantly Change the Curves?  (α = 0.05)")
    ax = fig.add_axes([0.12, 0.10, 0.80, 0.74])
    ax.set_facecolor(BG)

    # Separate colormaps per column type
    img_data = np_.zeros_like(data)
    for c in [0, 2]:   # delta columns: diverging around 0
        vmax = np_.abs(data[:, c]).max() + 0.01
        img_data[:, c] = data[:, c] / vmax * 0.5 + 0.5
    for c in [1, 3]:   # p-value columns: 0=significant (green), 1=n.s. (red)
        img_data[:, c] = data[:, c]

    cmap_delta = plt.cm.RdYlGn   # green=negative delta (improvement), red=positive
    cmap_pval  = plt.cm.RdYlGn_r  # green=low p (significant), red=high p

    cell_h, cell_w = 1.0, 1.0
    for r in range(len(METHODS_ALL)):
        for c in range(4):
            val = data[r, c]
            if c in [1, 3]:
                col = cmap_pval(min(val / 0.1, 1.0))
                txt = f"{val:.3f}" + (" *" if val < 0.05 else "")
                txt_col = "white"
            else:
                norm_val = (val + 0.3) / 0.6
                col = cmap_delta(np_.clip(1 - norm_val, 0, 1))
                txt = f"{val:+.3f}"
                txt_col = "white"
            rect = plt.Rectangle((c * cell_w, (len(METHODS_ALL)-1-r) * cell_h),
                                  cell_w, cell_h, facecolor=col, edgecolor=BG, linewidth=0.5)
            ax.add_patch(rect)
            ax.text(c * cell_w + 0.5, (len(METHODS_ALL)-1-r) * cell_h + 0.5, txt,
                    ha="center", va="center", fontsize=11, color=txt_col, fontweight="bold")

    ax.set_xlim(0, 4); ax.set_ylim(0, len(METHODS_ALL))
    ax.set_xticks([0.5, 1.5, 2.5, 3.5]); ax.set_xticklabels(cols, fontsize=11, color=FG)
    ax.set_yticks([i + 0.5 for i in range(len(METHODS_ALL))])
    ax.set_yticklabels(row_labels[::-1], fontsize=11, color=FG)
    ax.tick_params(length=0)
    ax.spines[:].set_visible(False)
    ax.text(0.5, -0.12, "* = p < 0.05  |  green = significant / improvement",
            ha="center", va="top", fontsize=10, color=FG, alpha=0.6,
            transform=ax.transAxes)

    save_slide(fig, 11, "significance-heatmap")


In [ ]:
import numpy as np_

ar = D.get("arch_results")
if ar is None:
    print("arch_results not available — skipping slide 12 (run Section 10 first)")
else:
    arch_names = list(ar.keys())
    fig = make_slide("Architecture Comparison: Calibration Generalizes Across Families")
    gs = fig.add_gridspec(1, 2, top=0.88, bottom=0.12,
                          left=0.06, right=0.97, wspace=0.28)

    tick_lbls, data_raw_auc, data_cal_auc = [], [], []
    data_raw_sp, data_cal_sp = [], []
    colors_raw, colors_cal = [], []
    arch_palette = [ACC1, "#9c88ff", ACC2, "#66bb6a"]
    for i, arch in enumerate(arch_names):
        data_raw_auc.append(ar[arch]["raw_aucs"])
        data_cal_auc.append(ar[arch]["cal_aucs"])
        data_raw_sp.append(ar[arch]["raw_sp"])
        data_cal_sp.append(ar[arch]["cal_sp"])
        tick_lbls += [f"{arch}\nRaw", f"{arch}\nCal"]
        colors_raw.append(arch_palette[i % len(arch_palette)] + "77")
        colors_cal.append(arch_palette[i % len(arch_palette)] + "cc")

    all_boxes_auc = []
    all_boxes_sp  = []
    all_colors    = []
    all_ticks     = []
    for i, arch in enumerate(arch_names):
        all_boxes_auc += [ar[arch]["raw_aucs"], ar[arch]["cal_aucs"]]
        all_boxes_sp  += [ar[arch]["raw_sp"],   ar[arch]["cal_sp"]]
        p = arch_palette[i % len(arch_palette)]
        all_colors += [p + "66", p + "cc"]
        all_ticks  += [f"{arch}\nRaw", f"{arch}\nCal"]

    for ax, boxes, ylabel, title in [
        (fig.add_subplot(gs[0]), all_boxes_auc, "AUC",       "Deletion AUC"),
        (fig.add_subplot(gs[1]), all_boxes_sp,  "Spearman r","Spearman r (deletion)"),
    ]:
        ax.set_facecolor("#12122a")
        bp = ax.boxplot(boxes, labels=all_ticks,
                        patch_artist=True,
                        medianprops=dict(color="white", linewidth=2))
        for patch, fc in zip(bp["boxes"], all_colors):
            patch.set_facecolor(fc)
        for elem in ["whiskers","caps","fliers"]:
            for line in bp[elem]: line.set(color=FG, alpha=0.4)
        ax.set_title(title, color=FG, fontsize=12)
        ax.set_ylabel(ylabel, color=FG, fontsize=10)
        ax.tick_params(colors=FG, labelsize=8)
        ax.spines[:].set_color(FG); ax.spines[:].set_alpha(0.25)
        ax.grid(axis="y", alpha=0.15, color=FG)

    save_slide(fig, 12, "arch-comparison")


In [ ]:
import numpy as np_

ar = D.get("arch_results")
if ar is None:
    print("arch_results not available — skipping slide 13")
else:
    arch_names = list(ar.keys())
    NUM_BINS_l = D.get("NUM_BINS", 10)
    bins_x = np_.arange(NUM_BINS_l)
    arch_palette = [ACC1, "#9c88ff", ACC2, "#66bb6a"]

    fig = make_slide("Temperature Profiles Vary by Architecture and Perturbation Depth")
    nrows, ncols = 2, 2
    axes = fig.subplots(nrows, ncols)
    fig.subplots_adjust(top=0.88, bottom=0.08, left=0.08, right=0.96,
                        hspace=0.38, wspace=0.28)

    ymax = 0
    arch_temps = {arch: ar[arch]["temps"] for arch in arch_names if "temps" in ar[arch]}

    for ax, arch, col in zip(axes.flat, arch_names, arch_palette):
        temps = arch_temps.get(arch, [1.0]*NUM_BINS_l)
        ymax = max(ymax, max(temps) + 0.15)

    for ax, arch, col in zip(axes.flat, arch_names, arch_palette):
        ax.set_facecolor("#12122a")
        temps = arch_temps.get(arch, [1.0]*NUM_BINS_l)
        ax.bar(bins_x, temps[:NUM_BINS_l], color=col + "99", edgecolor=BG, linewidth=0.5)
        ax.axhline(1.0, color=ACC2, linewidth=1.5, linestyle="--", alpha=0.8)
        ax.set_title(arch, color=col, fontsize=11, fontweight="bold")
        ax.set_ylim(0, max(ymax, 2.0))
        ax.set_xticks(bins_x[::2])
        ax.set_xticklabels([f"{b/NUM_BINS_l:.1f}" for b in range(0, NUM_BINS_l, 2)],
                           fontsize=7, color=FG)
        ax.tick_params(colors=FG, labelsize=8)
        ax.spines[:].set_color(FG); ax.spines[:].set_alpha(0.25)
        ax.set_xlabel("Perturbation level", color=FG, fontsize=8)
        ax.set_ylabel("T", color=FG, fontsize=8)
        ax.grid(axis="y", alpha=0.15, color=FG)

    save_slide(fig, 13, "temp-profiles")


In [ ]:
fig = make_slide("Future Work")
ax = fig.add_axes([0, 0.05, 1, 0.85])
ax.set_facecolor(BG); ax.axis("off")
ax.set_xlim(0, 10); ax.set_ylim(0, 5)

# Left: OOD diagram
boxes = [
    (1.8, 3.8, "Occlusion /\nIntGrad", "#8a6ff0"),
    (1.8, 2.2, "Modified input\n(zeroed / interp.)", ACC2),
    (1.8, 0.8, "OOD →\nMiscalibrated", ACC2),
    (7.5, 2.2, "Apply ReCalX\nto explanation", ACC3),
]
for (x, y, lbl, col) in boxes:
    rect = mpatches.FancyBboxPatch((x-1.1, y-0.45), 2.2, 0.9,
                                   boxstyle="round,pad=0.08",
                                   edgecolor=col, facecolor=col+"28", linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, y, lbl, ha="center", va="center", fontsize=9,
            color=col, fontweight="bold")
for (x1,y1),(x2,y2) in [((1.8,3.35),(1.8,2.65)),((1.8,1.75),(1.8,1.25)),
                          ((2.9,2.2),(6.4,2.2))]:
    ax.annotate("", xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle="-|>", color=FG, lw=1.2, alpha=0.5))

# Right: bullet points
bullets = [
    ("Calibrate Occlusion & IG explanations", ACC3),
    ("Test on full ImageNet validation set", ACC1),
    ("Inpainting-based deletion  (LaMa)", ACC1),
    ("Temporal calibration for video models", ACC1),
]
for i, (txt, col) in enumerate(bullets):
    y = 3.8 - i * 0.9
    circle = plt.Circle((5.5, y), 0.15, color=col, zorder=3)
    ax.add_patch(circle)
    ax.text(5.85, y, txt, va="center", fontsize=13, color=FG)

save_slide(fig, 14, "future-work")


In [ ]:
fig = make_slide("Summary")
ax = fig.add_axes([0.05, 0.06, 0.90, 0.84])
ax.set_facecolor(BG); ax.axis("off")
ax.set_xlim(0, 10); ax.set_ylim(0, 7)

takeaways = [
    (ACC2,
     "Deletion curves are systematically miscalibrated",
     "Model confidence ≠ accuracy at any perturbation level  —  OOD images create position-dependent bias"),
    (RAW_C,
     "ReCalX reduces ECE by ~38%, TACE by ~48%",
     "Consistent improvement across 6 attribution methods, 2 generator strategies, and 4 architectures"),
    (ACC3,
     "LIME best identifies salient pixels; Bucket deletion most monotone",
     "Spearman r and monotonicity ratio expose curve quality beyond AUC; calibration benefit is statistically significant for most methods"),
]
for i, (col, headline, subtext) in enumerate(takeaways):
    y_top = 5.8 - i * 2.2
    rect = mpatches.FancyBboxPatch((0.3, y_top - 1.0), 9.4, 1.55,
                                   boxstyle="round,pad=0.12",
                                   edgecolor=col, facecolor=col+"1a", linewidth=2)
    ax.add_patch(rect)
    ax.text(5.0, y_top + 0.35, headline,
            ha="center", va="center", fontsize=14, fontweight="bold", color=col)
    ax.text(5.0, y_top - 0.30, subtext,
            ha="center", va="center", fontsize=9.5, color=FG, alpha=0.85,
            wrap=True)

save_slide(fig, 15, "summary")
print("\n=== All 15 slides saved to docs/presentation_slides/ ===")

In [ ]:
import pathlib
slides = sorted(pathlib.Path("../docs/presentation_slides").glob("slide_*.png"))
print(f"Generated {len(slides)} slides:")
for s in slides:
    print(f"  {s.name}")
